<a href="https://colab.research.google.com/github/yoshevijay/Product-review-acknowledgement-/blob/main/Customer_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title
import pandas as pd
import re
import random
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import spacy
from spacy.lang.en.stop_words import STOP_WORDS as stopwords

In [ ]:
!conda install -y gdown

/bin/bash: line 1: conda: command not found


In [ ]:
!gdown --id 1s-8A8sF7b23Tb9Myoc_3DTl6YXLpL17L  # train data
!gdown --id 1YtAHCzeZUXGZQ9cimdkkUq4lUk3ZH-I_  # evaluate.py



/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1s-8A8sF7b23Tb9Myoc_3DTl6YXLpL17L
To: /content/lab4_train.csv
100% 331k/331k [00:00<00:00, 88.3MB/s]
/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1YtAHCzeZUXGZQ9cimdkkUq4lUk3ZH-I_
From (redirected): https://drive.google.com/uc?id=1YtAHCzeZUXGZQ9cimdkkUq4lUk3ZH-I_&confirm=t&uuid=68a27949-d61b-4d35-8e21-b0226095f5cc
To: /content/evaluate.py
100% 7.03k/7.03k [00:00<00:00, 23.5MB/s]


In [ ]:
data = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
import re
stopwords = set(stopwords)
if 'Not' in stopwords:
    stopwords.remove('Not')
data['text'] = data['text'].apply(lambda x: ' '.join([t for t in x.split() if t not in stopwords]))
data['text'] = data['text'].apply(lambda x: re.sub(r'[^\w]', " ", x))

In [ ]:
X = data['text']
Y = data['polarity']

In [ ]:
pipelines = {
    'SVM': Pipeline([('tfidf', TfidfVectorizer()), ('clf', SVC(C=1000, gamma='auto'))]),
    'Decision Tree': Pipeline([('tfidf', TfidfVectorizer()), ('clf', DecisionTreeClassifier())]),
    'Random Forest': Pipeline([('tfidf', TfidfVectorizer()), ('clf', RandomForestClassifier())]),
    'Logistic Regression': Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression())])
}

In [ ]:

for name, clf in pipelines.items():
    # Compute cross-validated scores
    scores = cross_val_score(clf, X, Y, cv=5)
    print(f"{name} Cross-validated Accuracy:", scores.mean())

# Choose the best classifier based on cross-validated scores
best_clf_name = max(pipelines, key=lambda k: cross_val_score(pipelines[k], X, Y, cv=5).mean())
best_clf = pipelines[best_clf_name]
print(f"Best Classifier: {best_clf_name}")

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
# Training loop with epochs (for the best model)
epochs =

for epoch in range(epochs):
    best_clf.fit(X_train, Y_train)

    # Optional evaluation on validation set (uncomment to evaluate)
    # Y_pred_val = best_clf.predict(X_val)
    # print(f"Epoch {epoch+1} Validation Performance Metrics (code not shown)")

    print(f"Epoch {epoch+1} complete")

# Make predictions and evaluate on training data
Y_predict = best_clf.predict(X)
print("Confusion Matrix:")
print(confusion_matrix(Y, Y_predict))
print("Classification Report:")
print(classification_report(Y, Y_predict))

In [ ]:
clf.predict(['I did not like the session conducted'])

**ASPECT BASED SENTIMENT ANALYSIS**

In [ ]:
def split_by_bar(string_text):
    return string_text.split(' ') #split data (=tokenize)

data['tokens'] = data['text'].apply(split_by_bar)

In [ ]:
data['tokens']
data['tokens'][10]

In [ ]:
data['tokens']

In [ ]:
data.head()

In [ ]:
data['aspectCategory']

In [ ]:
train, dev = np.split(data, [int(len(data)*0.8)])

In [ ]:
dev.shape

In [ ]:
train.shape

In [ ]:
train.to_csv('aspect-train.csv')
dev.to_csv('aspect-dev.csv')

In [ ]:
train

In [ ]:
dev

In [ ]:
from collections import Counter
from nltk.util import ngrams
n_gram = 1

In [ ]:
def unigram(lst):
  unigram = Counter(ngrams(lst, n_gram))
  return unigram

In [ ]:
train['ug'] = (train['tokens']).apply(unigram)

In [ ]:
train['ug']

In [ ]:
def featurize(token_list):
  features = {} # add output into dict = dict of features = list of feature dict
  for i in token_list:
    features[i] = 1
  return features

In [ ]:
feature_dict_list = [featurize(x) for x in train['ug']]

In [ ]:
feature_dict_list

In [ ]:
train

In [ ]:
from sklearn.feature_extraction import DictVectorizer
dv = DictVectorizer(sparse=True)
train_sparse_matrix = dv.fit_transform(feature_dict_list)

In [ ]:
train_sparse_matrix

In [ ]:
lm = LogisticRegression()
lm.fit(train_sparse_matrix, train['aspectCategory'])

**Train to predict category**

In [ ]:
dev['ug'] = (dev['tokens']).apply(unigram)

In [ ]:
dev

In [ ]:
# instance of vectorizer
vectorizer1 = DictVectorizer(sparse=True)

# make binary feature dictionary of train/dev data to make it more suitable
feature_dic_train1 = train['ug'].apply(featurize)
feature_dic_dev1 = dev['ug'].apply(featurize)

# convert to feature vectors
feature_vec_train1 = vectorizer1.fit_transform(feature_dic_train1)
feature_vec_dev1 = vectorizer1.transform(feature_dic_dev1)

# instance of Logistic Regression Model
classifier1 = LogisticRegression()

# train model with train data
classifier1.fit(feature_vec_train1, train['aspectCategory'])

In [ ]:
predict_result_aspect = classifier1.predict(feature_vec_dev1)

In [ ]:
train['predicted_category'] = classifier1.predict(feature_vec_train1)

In [ ]:
train['predicted_category']

In [ ]:
train.head()

In [ ]:
lm = LogisticRegression()
lm.fit(train_sparse_matrix, train['polarity'])

In [ ]:
dev_sparse_matrix

In [ ]:
predictions = lm.predict(dev_sparse_matrix)

**Train to predict polarity**

In [ ]:
vectorizer1 = DictVectorizer(sparse=True)

# make binary feature dictionary of train/dev data
feature_dic_train1 = train['ug'].apply(featurize)
feature_dic_dev1 = dev['ug'].apply(featurize)

# convert to feature vectors (use .fit_transform/.transform)
feature_vec_train1 = vectorizer1.fit_transform(feature_dic_train1)
feature_vec_dev1 = vectorizer1.transform(feature_dic_dev1)

# instance of Logistic Regression Model
classifier1 = LogisticRegression()

# train model with train data
classifier1.fit(feature_vec_train1, train['polarity'])

In [ ]:
predict_result_polarity = classifier1.predict(feature_vec_dev1)

**predict & create new df**

In [ ]:
 predict_result_aspect.shape

In [ ]:
 predict_result_polarity.shape

In [ ]:
newdf = pd.DataFrame()
newdf['id'] = dev['id']
newdf['aspectCategory'] = predict_result_aspect
newdf['polarity'] = predict_result_polarity
newdf

In [ ]:
# export to csv
newdf.to_csv('pred.csv', index=None)

In [ ]:
!python3 evaluate.py train.csv pred.csv